In [4]:
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional

I0000 00:00:1778612640.792537  140329 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778612640.832523  140329 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778612641.653125  140329 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [10]:
df = pd.read_csv("cleaned_twitter_data.csv")

In [ ]:
df.head()

,target,text,cleaned_text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",awww that bummer shoulda got david carr third day
1,0,is upset that he can't update his Facebook by ...,upset cant updat facebook text might cri resul...
2,0,@Kenichan I dived many times for the ball. Man...,dive mani time ball manag save rest go bound
3,0,my whole body feels itchy and like its on fire,whole bodi feel itchi like fire
4,0,"@nationwideclass no, it's not behaving at all....",no not behav im mad cant see


In [ ]:
df_small = df.sample(10000, random_state=42)

In [ ]:
df_small.shape

(10000, 3)

In [11]:
max_words = 50000
max_len = 150

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df["cleaned_text"])

sequences = tokenizer.texts_to_sequences(df["cleaned_text"])

In [4]:
sequences , len(sequences)

([[380, 51, 1068, 2995, 15, 713, 7213, 1637, 5],
  [602, 14, 225, 448, 366, 207, 240, 964, 84, 12, 191, 1054],
  [3574, 226, 13, 869, 699, 508, 353, 2, 2744],
  [338, 659, 25, 2501, 8, 878],
  [10, 4, 3968, 1, 466, 14, 24],
  [4, 338, 1903],
  [32, 396],
  [90, 99, 13, 10, 24, 95, 101, 172, 172, 21, 1, 430, 16, 672],
  [673, 60],
  [2084],
  [1291, 339, 2582, 485, 1409],
  [15823, 783],
  [321, 1304, 31, 153, 11617, 1361, 2766],
  [766, 732, 383, 97, 124, 320],
  [1972, 104, 60, 2331, 4, 27, 72, 3461, 23833],
  [46, 15, 31, 19, 2056],
  [2402, 919, 1572, 139, 1437, 31, 642, 23834, 4384, 451, 4],
  [1223, 2245],
  [587, 73, 123, 17, 24, 1508, 9, 3120],
  [40, 518, 274, 2290, 1443, 274],
  [5, 60, 3, 41, 120],
  [20, 67, 115, 297, 218, 3086, 3104, 6959, 73, 10, 13, 522],
  [1254, 636, 565],
  [56, 4, 2, 29],
  [28806, 288, 45],
  [75, 115, 317, 83],
  [2, 240, 50, 31, 2960],
  [1, 53],
  [2737, 21, 7048, 113, 117, 7048, 117, 3, 466],
  [1777, 263, 2143, 720, 894, 3, 618, 299, 13],
  [192

In [5]:
X = pad_sequences(sequences, maxlen=max_len)
y = df["target"].values

X.shape, y.shape

((1592563, 150), (1592563,))

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
len(X_train), len(X_test)

(1274050, 318513)

In [8]:
model = Sequential([
    Embedding(max_words, 128, input_length=max_len),
    LSTM(128, return_sequences=True),
    Dropout(0.2),
    LSTM(64),
    Dropout(0.2),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid"),
])

model.build(input_shape=(None, max_len))

/home/sadik/miniconda3/envs/tfgpu/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
W0000 00:00:1778611159.818623  133696 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1778611159.824885  133696 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1778611159.935626  133696 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13213 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5060 Ti, pci bus id: 0000:01:00.0, compute capability: 12.0a
E0000 00:00:1778611160.330529  134023 ptx_compiler_helpers.cc:154] *** WARNING *** Invoking ptxas with v

In [9]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 150, 128)       │     6,400,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 150, 128)       │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 150, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,585,217 (25.12 MB)

 Trainable params: 6,585,217 (25.12 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

In [ ]:
history = model.fit(X_train, y_train, epochs=5, batch_size=256, validation_data=(X_test, y_test))

Epoch 1/5
4977/4977 ━━━━━━━━━━━━━━━━━━━━ 94s 19ms/step - accuracy: 0.7943 - loss: 0.4415 - val_accuracy: 0.8066 - val_loss: 0.4206
Epoch 2/5
4977/4977 ━━━━━━━━━━━━━━━━━━━━ 93s 19ms/step - accuracy: 0.8190 - loss: 0.3994 - val_accuracy: 0.8102 - val_loss: 0.4146
Epoch 3/5
4977/4977 ━━━━━━━━━━━━━━━━━━━━ 93s 19ms/step - accuracy: 0.8337 - loss: 0.3721 - val_accuracy: 0.8078 - val_loss: 0.4240
Epoch 4/5
4977/4977 ━━━━━━━━━━━━━━━━━━━━ 93s 19ms/step - accuracy: 0.8485 - loss: 0.3423 - val_accuracy: 0.8038 - val_loss: 0.4483
Epoch 5/5
4977/4977 ━━━━━━━━━━━━━━━━━━━━ 93s 19ms/step - accuracy: 0.8632 - loss: 0.3121 - val_accuracy: 0.7981 - val_loss: 0.4846


In [ ]:
model.save("lstm_sentiment_model_train-accu-86_val-acc-81.h5")

In [11]:
model_bidirectional = Sequential([
    Embedding(max_words, 128, input_length=max_len),
    Bidirectional(LSTM(128)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

model_bidirectional.build(input_shape=(None, max_len))

In [12]:
model_bidirectional.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

In [13]:
model_bidirectional.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 150, 128)       │     6,400,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 256)            │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,679,681 (25.48 MB)

 Trainable params: 6,679,681 (25.48 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
callback = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

In [15]:
history_bidirectional = model_bidirectional.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=256,
    validation_data=(X_test, y_test),
    callbacks=[callback]
)

Epoch 1/5


W0000 00:00:1778611189.082107  134317 gpu_kernel_to_blob_pass.cc:190] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
W0000 00:00:1778611189.082537  134320 gpu_kernel_to_blob_pass.cc:190] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
W0000 00:00:1778611189.083538  134309 gpu_kernel_to_blob_pass.cc:190] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
W0000 00:00:1778611189.084700  134314 gpu_kernel_to_blob_pass.cc:190] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
W0000 00:00:1778611189.085621  134312 gpu_kernel_to_blob_pass.cc:190] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
W0000 00:00:1778611189.086661  134310 gpu_kernel_to_blob_pass.cc:190] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
W0000 00:00:1778611189.087568  134319 gpu_kernel_to_blob_pass.cc:190] Failed to co

   3/4977 ━━━━━━━━━━━━━━━━━━━━ 2:25 29ms/step - accuracy: 0.5256 - loss: 0.6927 

W0000 00:00:1778611191.618538  134011 gpu_kernel_to_blob_pass.cc:190] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.


4977/4977 ━━━━━━━━━━━━━━━━━━━━ 162s 32ms/step - accuracy: 0.7936 - loss: 0.4422 - val_accuracy: 0.8075 - val_loss: 0.4187
Epoch 2/5
4977/4977 ━━━━━━━━━━━━━━━━━━━━ 157s 32ms/step - accuracy: 0.8185 - loss: 0.3998 - val_accuracy: 0.8096 - val_loss: 0.4160
Epoch 3/5
4977/4977 ━━━━━━━━━━━━━━━━━━━━ 158s 32ms/step - accuracy: 0.8327 - loss: 0.3727 - val_accuracy: 0.8083 - val_loss: 0.4246
Epoch 4/5
4977/4977 ━━━━━━━━━━━━━━━━━━━━ 158s 32ms/step - accuracy: 0.8470 - loss: 0.3445 - val_accuracy: 0.8058 - val_loss: 0.4523


In [16]:
history_bidirectional.history

{'accuracy': [0.793647050857544,
  0.8184741735458374,
  0.8326863050460815,
  0.8470248579978943],
 'loss': [0.4421994686126709,
  0.39979901909828186,
  0.37270888686180115,
  0.34452754259109497],
 'val_accuracy': [0.8075180649757385,
  0.8096404075622559,
  0.8083280920982361,
  0.8057818412780762],
 'val_loss': [0.4187013804912567,
  0.4160307049751282,
  0.4246053695678711,
  0.4523424208164215]}

In [17]:
model_bidirectional.save("bidirectional_lstm_sentiment_model_train-accu-84_val-acc-80.h5")

In [5]:
model_bidirectional = keras.models.load_model("bidirectional_lstm_sentiment_model_train-accu-84_val-acc-80.h5")

W0000 00:00:1778612647.749040  140329 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1778612647.755647  140329 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1778612647.855320  140329 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13312 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5060 Ti, pci bus id: 0000:01:00.0, compute capability: 12.0a
E0000 00:00:1778612648.149439  141687 ptx_compiler_helpers.cc:154] *** WARNING *** Invoking ptxas with version 12.3.52, which corresponds to a CUDA version <=12.6.2. CUDA versions 12.x.y up to and including 12.6.2 miscompile certain edge cases around clamping.
Please upgrade to CUDA 12.6.3 

In [6]:
def predict_sentiment(text):
    sequence = tokenizer.texts_to_sequences([text])
    padded_sequence = pad_sequences(sequence, maxlen=max_len)
    prediction = model_bidirectional.predict(padded_sequence)[0][0]
    return "Positive" if prediction >= 0.5 else "Negative"

In [13]:
predict_sentiment("i have a exam")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step


'Negative'